In [1]:
pip install transformers[torch] datasets scikit-learn

In [5]:
import pandas as pd

# 1. Mount Drive (Hanya perlu sekali per sesi)
from google.colab import drive
drive.mount('/content/drive')

# 2. Baca file CSV
# Perhatikan penggunaan 'My Drive' (pakai spasi) setelah '/content/drive/'
path = '/content/drive/My Drive/CSV/reviews_cleaned.csv'
df = pd.read_csv(path)

# Cek data
df.head()

Mounted at /content/drive


,content_cleaned,label
0,dana cicil tidak bisa di pake padahal udah bay...,0.0
1,good,1.0
2,bagus,1.0
3,daptar belum bisa update,0.0
4,knp paylatter saya tidak bisa,1.0


In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset


# Perhatikan penggunaan 'My Drive' (pakai spasi) setelah '/content/drive/'
#path = '/content/drive/My Drive/CSV/reviews_cleaned.csv'
#df = pd.read_csv(path)

df['content_cleaned'] = df['content_cleaned'].fillna('') # Isi data kosong dengan string kosong
df['content_cleaned'] = df['content_cleaned'].astype(str) # Pastikan semua tipe data adalah string

# Ubah label ke format integer (0 dan 1)
df['label'] = df['label'].astype(int)

# 2. Split data: 80% Latihan, 20% Ujian
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

# 3. Load Tokenizer (Penerjemah kata ke angka untuk IndoBERT)
model_name = "indobenchmark/indobert-base-p1"
tokenizer = BertTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["content_cleaned"], padding="max_length", truncation=True, max_length=128)

# Ubah ke format Dataset Hugging Face
train_dataset = Dataset.from_dict(train_df).map(tokenize_function, batched=True)
val_dataset = Dataset.from_dict(val_df).map(tokenize_function, batched=True)

# 4. Load Model
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 5. Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",      # <--- Ganti dari evaluation_strategy menjadi eval_strategy
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
)

# 6. Mulai Training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("Memulai training... (Ini akan memakan waktu)")
trainer.train()

# 7. Simpan Modelnya
model.save_pretrained("./indobert-sentiment-model")
tokenizer.save_pretrained("./indobert-sentiment-model")
print("Model berhasil disimpan di folder 'indobert-sentiment-model'!")

Map:   0%|          | 0/384 [00:00<?, ? examples/s]

Map:   0%|          | 0/97 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Memulai training... (Ini akan memakan waktu)


Epoch,Training Loss,Validation Loss
1,No log,0.310375
2,No log,0.232491
3,No log,0.188898


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model berhasil disimpan di folder 'indobert-sentiment-model'!


In [9]:
import os

os.listdir('/content')

['.config', 'indobert-sentiment-model', 'results', 'drive', 'sample_data']

In [10]:
model.save_pretrained('/content/drive/MyDrive/ML_Project/indobert-sentiment-model')
tokenizer.save_pretrained('/content/drive/MyDrive/ML_Project/indobert-sentiment-model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/ML_Project/indobert-sentiment-model/tokenizer_config.json',
 '/content/drive/MyDrive/ML_Project/indobert-sentiment-model/tokenizer.json')